# Train your own TopK SAE on GraphCast activations

Qiansheng Han · AMS691 term project

This notebook trains the authors' existing `SAE` class on **real cached GraphCast activation vectors**. GraphCast stays fixed. We will later train KAN-SAE on the same data for a controlled comparison.

Use a separate Colab runtime for this training stage. You need floating-point `.npy` files from one hook (default `layer0008_`) spanning multiple dates. A GPU helps SAE training; the full GraphCast model is not loaded here.

[Training guide](https://github.com/observerhan/graphcast-interpretability/blob/topk-trainer/docs/topk_training.md) · [Original extraction demo](https://github.com/observerhan/graphcast-interpretability/blob/main/hurricane_feature_demo.ipynb)


## 1. Install the trainer

Run this notebook in Colab, or choose a Colab kernel in VS Code. The installation runs on the selected runtime and fetches the committed `topk-trainer` branch of your fork.


In [ ]:
%pip install -q "graphcast-interpretability[training] @ git+https://github.com/observerhan/graphcast-interpretability.git@topk-trainer"


In [ ]:
from pathlib import Path
import numpy as np
import torch

from graphcast_interpretability.train_topk import (
    TrainConfig, find_activation_files, split_by_day, iter_batches, train_topk,
)

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU fallback")


## 2. Connect Drive and set your paths

Keep both the activations and run outputs on Drive so they survive a runtime disconnect. In the extraction notebook, save activations numerically as float32 (`np.asarray(x).astype(np.float32)`). Legacy `|V2` arrays need verified conversion or re-extraction.

If extraction saved valid files in `/content/graphcast_acts`, you can copy that directory to Drive **from that extraction runtime** with `shutil.copytree(source, destination, dirs_exist_ok=True)` after mounting Drive. This training notebook requires those files to exist already.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# EDIT these paths to match your Drive folders.
ACTIVATION_DIR = "/content/drive/MyDrive/AMS691/graphcast_acts"
RUN_DIR = "/content/drive/MyDrive/AMS691/runs/topk_layer8_pilot"

# Fresh run: False. Restart after a disconnect: True, and increase TOTAL_EPOCHS.
RESUME = False
TOTAL_EPOCHS = 5


## 3. Check the data and hold out whole dates

The automatic split reserves the latest 20% of calendar days. It needs at least two distinct dates and timestamped filenames. One day of hurricane data alone is not enough. A later scientific comparison should hold out whole storms/events, with a separate final test set.

For explicit event splits, replace the next cell's `split_by_day(...)` call with separate `find_activation_files(TRAIN_DIR)` and `find_activation_files(VAL_DIR)` calls.


In [ ]:
files = find_activation_files(ACTIVATION_DIR, prefix="layer0008_")
train_files, val_files = split_by_day(files, val_fraction=0.2)

sample = np.load(files[0], mmap_mode="r", allow_pickle=False)
print("First activation file:", files[0].name)
print("Shape:", sample.shape, "dtype:", sample.dtype)
print("Training files:", len(train_files), "Validation files:", len(val_files))
print("First training file:", train_files[0].name)
print("First validation file:", val_files[0].name)


## 4. Configure the pilot

Each input has 512 coordinates; the SAE learns 4,096 features and retains at most 32 positive activations per input. `batch_size=256` is a small starting setting. Reduce it if memory is limited.

**Pilot choice:** `dead_window=100_000` makes the auxiliary recovery mechanism eligible sooner in a small dataset. The paper describes 5,000,000 vector presentations; the trainer's default matches that value. Record this pilot change before choosing the final comparison settings. The auxiliary weight is 1/32 and `k_aux=512`.

An epoch is one pass through the training vectors. Repeated epochs reuse the same vectors. This is a smaller training experiment than the paper's billions of vectors.


In [ ]:
config = TrainConfig(
    output_dir=RUN_DIR,
    d_in=512,
    latent=4096,
    k=32,
    k_aux=512,
    batch_size=256,
    epochs=TOTAL_EPOCHS,
    lr=3e-4,
    aux_weight=1/32,
    dead_window=100_000,
    seed=0,
    device="auto",
    log_every=50,
)

resume_path = str(Path(RUN_DIR) / "last.pt") if RESUME else None
result = train_topk(train_files, val_files, config, resume=resume_path)
print("Best checkpoint:", result["best_checkpoint"])


## 5. Inspect learning and sparsity

Reconstruction error is measured against the same centered, unit-length vectors reconstructed by the authors' class. The loss sums over 512 coordinates and averages over vectors.

`val_fvu < 1` beats predicting the training mean. Lower error is useful, but weather-feature interpretability must still be evaluated separately.


In [ ]:
import matplotlib.pyplot as plt

history = result["history"]
epochs = [row["epoch"] for row in history]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(epochs, [row["train_reconstruction_mse"] for row in history], label="Training")
axes[0].plot(epochs, [row["val_reconstruction_mse"] for row in history], label="Validation")
axes[0].axhline(result["initial_validation"]["reconstruction_mse"], color="gray", linestyle="--", label="Before training")
axes[0].set(xlabel="Epoch", ylabel="Squared reconstruction error per vector")
axes[0].legend()
axes[1].plot(epochs, [row["val_mean_l0"] for row in history], marker="o")
axes[1].axhline(config.k, color="gray", linestyle="--", label="TopK limit")
axes[1].set(xlabel="Epoch", ylabel="Mean active features per validation vector")
axes[1].legend()
fig.tight_layout()
plt.show()

print("Final validation FVU:", history[-1]["val_fvu"])
print("Validation feature coverage:", history[-1]["val_fraction_features_used"])
print("Run files:", [p.name for p in sorted(Path(RUN_DIR).iterdir())])


## 6. Reload the best SAE

This checkpoint uses the original demo's `model_state` key. It also includes the optimizer, dead-feature counters, split manifest, training settings, and epoch history.


In [ ]:
from graphcast_interpretability.model import SAE

checkpoint = torch.load(result["best_checkpoint"], map_location="cpu", weights_only=True)
cfg = checkpoint["config"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sae = SAE(cfg["d_in"], cfg["latent"], cfg["k"], cfg["k_aux"], dead_window=cfg["dead_window"])
sae.load_state_dict(checkpoint["model_state"], strict=True)
sae = sae.to(device).eval()

raw_batch = next(iter_batches(val_files, cfg["d_in"], 128)).to(device)
with torch.no_grad():
    reconstruction, codes, _ = sae(raw_batch)
print("Input shape:", tuple(raw_batch.shape))
print("Feature shape:", tuple(codes.shape))
print("Mean positive features:", (codes > 0).sum(dim=1).float().mean().item())


## Resume or start another experiment

After a disconnect: reinstall this same branch, remount Drive, use the same input files and `RUN_DIR`, set `RESUME=True`, increase `TOTAL_EPOCHS` (for example from 5 to 10), and rerun the remaining cells. Resume uses the last completed epoch; a partial interrupted epoch is repeated.

For a new seed or new settings: choose a new `RUN_DIR` and set `RESUME=False`. Existing runs are protected against accidental overwrite.

For the eventual KAN-SAE comparison, retain the exact training/validation split and preprocessing. Match dictionary size, compare across comparable effective sparsities, and record compute and data budgets. Select features using training/validation events, then evaluate on a separate test set.
